# 🎬 CineMatch — Movie Recommendation System
**Capstone | TF-IDF + Cosine Similarity**

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn ipywidgets -q
print('Libraries ready!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')
print('Imports done!')

In [ ]:
data = {
    'title': ['The Dark Knight','Inception','Interstellar','The Matrix','Avengers: Endgame',
              'Pulp Fiction','The Godfather','Forrest Gump','The Shawshank Redemption','Fight Club',
              'Joker','Dune','Mad Max: Fury Road','Deadpool','Logan','Get Out','A Quiet Place',
              'Doctor Strange','Guardians of the Galaxy','Black Panther','Top Gun: Maverick',
              'John Wick','Goodfellas','The Silence of the Lambs','The Avengers','Iron Man',
              'Spider-Man','Wonder Woman','Ant-Man','Avengers: Infinity War'],
    'genres': ['Action Crime Drama','Action Sci-Fi Thriller','Adventure Drama Sci-Fi',
               'Action Sci-Fi','Action Adventure Sci-Fi','Crime Drama Thriller','Crime Drama',
               'Drama Romance','Drama','Drama Thriller','Crime Drama Thriller',
               'Adventure Drama Sci-Fi','Action Adventure Sci-Fi','Action Comedy Sci-Fi',
               'Action Drama Sci-Fi','Horror Mystery Thriller','Drama Horror Sci-Fi',
               'Action Adventure Fantasy','Action Adventure Comedy Sci-Fi','Action Adventure Sci-Fi',
               'Action Drama Thriller','Action Crime Thriller','Biography Crime Drama',
               'Crime Drama Thriller','Action Adventure Sci-Fi','Action Adventure Sci-Fi',
               'Action Adventure Sci-Fi','Action Adventure Fantasy','Action Adventure Comedy Sci-Fi',
               'Action Adventure Sci-Fi'],
    'keywords': ['batman gotham joker dark superhero','dreams heist subconscious reality layers',
                 'space wormhole black hole gravity love','hacker simulation ai rebellion machines',
                 'infinity stones time travel heroes snap','hitman gangster crime redemption nonlinear',
                 'mafia family power crime loyalty','destiny love vietnam simple life running',
                 'prison hope friendship redemption wrongful','soap anarchy identity fight masculinity',
                 'clown chaos society origin mental health','desert planet prophecy sand worm spice',
                 'post-apocalyptic desert car chase fury road','mercenary regeneration comedy fourth wall',
                 'mutant claws aging western dark future','racism doppelganger horror sunken place',
                 'silence creature family survival monster','sorcerer magic dimensions mystical ancient',
                 'space team raccoon tree music cosmic funny','king africa vibranium wakanda royalty',
                 'pilot jet fighter maverick nostalgia training','assassin revenge hitman dog unstoppable',
                 'mob crime loyalty betrayal rise fall henry','serial killer fbi profiling cannibalism lecter',
                 'superhero team alien battle earth loki','billionaire suit arc reactor genius technology',
                 'teenager web radioactive spider mutation','warrior goddess amazon war love justice',
                 'small hero shrink science heist ant technology','thanos infinity stones snap universe heroes'],
    'overview': ['Batman raises the stakes against the Joker who creates chaos in Gotham',
                 'A thief plants an idea in a targets mind through dream sharing technology',
                 'Explorers travel through a wormhole searching for a new home for humanity',
                 'A programmer discovers reality is a simulation and joins a machine rebellion',
                 'The Avengers undo the Thanos snap by travelling through time',
                 'Two hitmen a boxer and a gangster intertwine in four crime tales',
                 'An aging crime patriarch transfers power to his reluctant son',
                 'A low IQ man from Alabama witnesses defining historical events',
                 'Two prisoners bond and find redemption through decency over the years',
                 'An office worker forms an underground fight club with a soap salesman',
                 'A failed comedian descends into madness and becomes the Joker',
                 'A noble family fights for control of a desert planet with precious spice',
                 'A woman flees a tyrannical ruler across a post-apocalyptic wasteland',
                 'An immortal mercenary seeks revenge while breaking the fourth wall',
                 'An aging Wolverine protects a young mutant girl in a dark future',
                 'A man visits his white girlfriend family and uncovers a dark secret',
                 'A family hides from creatures that hunt entirely by sound',
                 'A surgeon becomes the Master of Mystic Arts to protect Earth',
                 'A group of space criminals band together to save the universe',
                 'The king of Wakanda defends his nation and its vibranium resources',
                 'A pilot trains graduates while confronting the ghosts of his past',
                 'A retired hitman seeks revenge on gangsters who killed his dog',
                 'Henry Hill rises through the New York Mob before everything collapses',
                 'An FBI trainee works with cannibal Hannibal Lecter to catch a killer',
                 'Earths heroes team up to stop an alien invasion led by Loki',
                 'A billionaire builds an iron suit after being held captive in a cave',
                 'A teenager bitten by a radioactive spider gains extraordinary powers',
                 'An Amazon princess leaves her island to fight in World War One',
                 'A thief uses a shrinking suit to pull off a heist to save the world',
                 'The Avengers battle Thanos who wants to wipe out half all life'],
    'vote_average': [9.0,8.8,8.7,8.7,8.4,8.9,9.2,8.8,9.3,8.8,8.5,8.1,8.1,8.0,8.1,7.7,7.5,7.5,8.0,7.3,8.3,7.4,8.7,8.6,8.0,7.9,7.4,7.4,7.3,8.4],
    'vote_count': [32000,35000,33000,25000,22000,28000,18000,25000,26000,29000,24000,22000,18000,20000,14000,14000,16000,14000,19000,16000,16000,14000,12000,16000,25000,22000,15000,15000,12000,25000]
}

df = pd.DataFrame(data)
print(f'Dataset loaded: {len(df)} movies')
df.head()

In [ ]:
# Data Wrangling
df['genres']   = df['genres'].fillna('')
df['keywords'] = df['keywords'].fillna('')
df['overview'] = df['overview'].fillna('')
df['soup'] = df['genres'].str.lower() + ' ' + df['keywords'].str.lower() + ' ' + df['overview'].str.lower()
C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.70)
df['weighted_rating'] = ((df['vote_count']/(df['vote_count']+m))*df['vote_average'] + (m/(df['vote_count']+m))*C).round(2)
print('Data wrangling complete!')
print(df[['title','vote_average','weighted_rating']].head())

In [ ]:
# ML Algorithm: TF-IDF + Cosine Similarity
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['soup'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()
print(f'TF-IDF Matrix: {tfidf_matrix.shape}')
print('Model ready!')

In [ ]:
def get_recommendations(title, top_n=5):
    title_lower = title.strip().lower()
    if title_lower not in indices:
        matches = [t for t in indices.index if title_lower in t]
        if not matches:
            print(f'Movie not found: {title}')
            return None
        title_lower = matches[0]
    idx = indices[title_lower]
    sim_scores = sorted(list(enumerate(cosine_sim[idx])), key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    movie_indices = [s[0] for s in sim_scores]
    result = df.iloc[movie_indices][['title','genres','vote_average','weighted_rating']].copy()
    result['similarity_%'] = [round(s[1]*100,1) for s in sim_scores]
    result.index = range(1, len(result)+1)
    return result

print(get_recommendations('Inception'))

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0a0a0f')
for ax in axes.flat:
    ax.set_facecolor('#17171f')
    ax.tick_params(colors='#aaaaaa')
    for spine in ax.spines.values(): spine.set_edgecolor('#2a2a3a')

top10 = df.nlargest(10,'weighted_rating')
axes[0,0].barh(top10['title'], top10['weighted_rating'], color='#e8b86d')
axes[0,0].set_title('Top 10 Movies', color='white', fontweight='bold')
axes[0,0].set_xlim(8,9.5)

gc = {}
for g in df['genres']:
    for x in g.split(): gc[x] = gc.get(x,0)+1
gc = dict(sorted(gc.items(), key=lambda x: x[1], reverse=True)[:8])
axes[0,1].bar(gc.keys(), gc.values(), color='#7c6af4')
axes[0,1].set_title('Genre Distribution', color='white', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=30)

axes[1,0].hist(df['vote_average'], bins=10, color='#7c6af4', edgecolor='#0a0a0f')
axes[1,0].axvline(df['vote_average'].mean(), color='#e8b86d', linestyle='--', linewidth=2)
axes[1,0].set_title('Rating Distribution', color='white', fontweight='bold')

top8 = df.nlargest(8,'weighted_rating').index
sub = cosine_sim[np.ix_(top8, top8)]
im = axes[1,1].imshow(sub, cmap='magma', vmin=0, vmax=1)
labels = [df.loc[i,'title'][:12] for i in top8]
axes[1,1].set_xticks(range(8)); axes[1,1].set_yticks(range(8))
axes[1,1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1,1].set_yticklabels(labels, fontsize=7)
axes[1,1].set_title('Similarity Heatmap', color='white', fontweight='bold')
plt.colorbar(im, ax=axes[1,1])

plt.suptitle('CineMatch — Dataset Analytics', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('Visualization saved!')

In [ ]:
# Interactive GUI
display(HTML('<h2 style="color:#e8b86d">🎬 CineMatch — Movie Recommender</h2>'))
dd = widgets.Dropdown(options=sorted(df['title'].tolist()), value='Inception', description='Movie:', layout=widgets.Layout(width='350px'))
sl = widgets.IntSlider(value=5, min=3, max=10, description='Top N:', layout=widgets.Layout(width='300px'))
btn = widgets.Button(description='Get Recommendations', button_style='info', layout=widgets.Layout(width='200px'))
out = widgets.Output()
def on_click(b):
    with out:
        clear_output(wait=True)
        r = get_recommendations(dd.value, sl.value)
        if r is None: return
        html = f'<h3 style="color:#f0eefc">Because you liked <span style="color:#e8b86d">{dd.value}</span>:</h3>'
        for i, row in r.iterrows():
            html += f'<div style="background:#17171f;border:1px solid #2a2a3a;border-left:3px solid #e8b86d;border-radius:10px;padding:12px;margin:8px 0"><b style="color:#f0eefc">#{i} {row["title"]}</b><br><span style="color:#6b6880">{row["genres"]} | ⭐ {row["vote_average"]}</span><br><div style="background:#2a2a3a;height:5px;border-radius:3px;margin-top:8px"><div style="background:linear-gradient(90deg,#7c6af4,#e8b86d);height:5px;width:{row["similarity_%"]}%;border-radius:3px"></div></div><small style="color:#e8b86d">{row["similarity_%"]}% match</small></div>'
        display(HTML(html))
btn.on_click(on_click)
display(widgets.HBox([dd, sl]))
display(btn, out)
on_click(None)